# Spatial Join: Tucson Streetlights × Council Districts

This notebook performs a spatial join to count streetlights per city council district.

**Required data files** (place in `data/` folder):
- `data/Tucson_Streetlights.geojson`
- `data/Tucson_Council_Districts.geojson`

See `data/README.md` for download instructions.

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

## Load Data

In [ ]:
streetlights = gpd.read_file('data/Tucson_Streetlights.geojson')
districts = gpd.read_file('data/Tucson_Council_Districts.geojson')

print('Streetlights CRS:', streetlights.crs)
print('Districts CRS:', districts.crs)
print('\nStreetlights:', len(streetlights), 'features')
print('Districts:', len(districts), 'features')

## Inspect the Data

In [ ]:
streetlights.head(3)

In [ ]:
districts.head(3)

## Match CRS

Both layers must be in the same CRS before we can run `sjoin`.

In [ ]:
if streetlights.crs != districts.crs:
    streetlights = streetlights.to_crs(districts.crs)
    print('Reprojected streetlights to match districts CRS')
else:
    print('CRS already matches:', districts.crs)

## Spatial Join

`sjoin` with `predicate='within'` assigns each streetlight point to the district polygon it falls inside.

In [ ]:
joined = gpd.sjoin(streetlights, districts, how='inner', predicate='within')
print('Joined rows:', len(joined))
joined.head(3)

## Count Streetlights per District

In [ ]:
# Identify the district name/ID column — update if your data uses a different column name
district_col = [c for c in districts.columns if 'district' in c.lower() or 'name' in c.lower()][0]
print('Using column:', district_col)

counts = joined.groupby(district_col).size().reset_index(name='streetlight_count')
counts = counts.sort_values('streetlight_count', ascending=False)
print(counts)

## Merge Counts Back to Districts

In [ ]:
districts_counted = districts.merge(counts, on=district_col, how='left')
districts_counted['streetlight_count'] = districts_counted['streetlight_count'].fillna(0).astype(int)
districts_counted[[district_col, 'streetlight_count']].head()

## Choropleth Map — Streetlights per District

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

districts_counted.plot(
    column='streetlight_count',
    cmap='YlOrBr',
    linewidth=1,
    edgecolor='black',
    legend=True,
    legend_kwds={'label': 'Streetlight Count', 'orientation': 'horizontal'},
    ax=ax
)

# Label each district
for _, row in districts_counted.iterrows():
    centroid = row.geometry.centroid
    ax.annotate(
        f"{row[district_col]}\n({row['streetlight_count']})",
        xy=(centroid.x, centroid.y),
        ha='center', fontsize=8, color='black'
    )

ax.set_title('Tucson Streetlights by Council District', fontsize=14)
ax.set_axis_off()
plt.tight_layout()
plt.show()

## Bar Chart

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
counts_sorted = counts.sort_values('streetlight_count')
ax.barh(counts_sorted[district_col].astype(str), counts_sorted['streetlight_count'], color='steelblue')
ax.set_xlabel('Number of Streetlights')
ax.set_title('Streetlights per Council District')
plt.tight_layout()
plt.show()